In [1]:
!nvidia-smi

Thu Apr  9 09:29:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from numba import cuda
import numpy as np

In [3]:
import torch

print(torch.cuda.get_device_name(0))

Tesla T4


In [4]:
import torch

props = torch.cuda.get_device_properties(0)
print(props.max_threads_per_block)

1024


In [6]:
import tensorflow as tf

print(tf.config.list_physical_devices('GPU'))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [7]:
import torch

props = torch.cuda.get_device_properties(0)

print("GPU:", props.name)
print("Max threads/block:", props.max_threads_per_block)
print("Multiprocessors:", props.multi_processor_count)

GPU: Tesla T4
Max threads/block: 1024
Multiprocessors: 40


In [8]:
from numba import cuda
import numpy as np

# CUDA kernel
@cuda.jit
def reduce_sum(arr):
    tid = cuda.threadIdx.x

    # Parallel reduction
    stride = 256
    while stride > 0:
        if tid < stride:
            arr[tid] += arr[tid + stride]
        cuda.syncthreads()
        stride //= 2

# Host data: numbers 1 to 512
data = np.arange(1, 513, dtype=np.int32)

# Copy to GPU
d_data = cuda.to_device(data)

# Launch kernel: 1 block, 512 threads
reduce_sum[1, 512](d_data)

# Copy back result
result = d_data.copy_to_host()

print("Sum =", result[0])

/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


Sum = 131328
